# Detección de Neumonía — 9 Arquitecturas + Análisis de Estabilidad

**Dos mejoras en una corrida:**
1. **9 arquitecturas** (se añaden ConvNeXt-Tiny, Swin-T y EfficientNetV2-S a las 6 originales)
2. **Múltiples semillas** (3 mejores modelos × 3 semillas → media ± desviación estándar)

| Fase | Qué hace | Tiempo aprox. |
|---|---|---|
| A | 9 arquitecturas, semilla 42 | ~1.5 h |
| B | Top-3 × semillas {42, 123, 2024} | ~1.5 h |
| C | Ensamblado + tablas + guardado | minutos |

**Instrucciones:** GPU T4, ejecutar en orden, subir `kaggle.json` cuando lo pida.

In [ ]:
!pip install -q kaggle timm scikit-learn
import torch
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.is_available())

In [ ]:
from google.colab import files
import os
print('Sube tu kaggle.json:')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
for fn in uploaded.keys(): os.rename(fn, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/data --unzip
print('Dataset listo')

In [ ]:
# Partición binaria 70/15/15 — idéntica a los experimentos previos (semilla 42 para la partición)
import shutil, random
from pathlib import Path
random.seed(42)
SRC=Path('/content/data/chest_xray')
ALL={'NORMAL':[],'PNEUMONIA':[]}
for split in ['train','test','val']:
    sd=SRC/split
    if not sd.exists(): continue
    for img in (sd/'NORMAL').glob('*.jpeg'): ALL['NORMAL'].append(img)
    for img in (sd/'PNEUMONIA').glob('*.jpeg'): ALL['PNEUMONIA'].append(img)
DEST=Path('/content/dataset_bin')
if DEST.exists(): shutil.rmtree(DEST)
for cls,imgs in ALL.items():
    imgs=imgs.copy(); random.shuffle(imgs)
    n=len(imgs); ntr=int(n*0.70); nv=int(n*0.15)
    for sn,si in {'train':imgs[:ntr],'val':imgs[ntr:ntr+nv],'test':imgs[ntr+nv:]}.items():
        od=DEST/sn/cls; od.mkdir(parents=True,exist_ok=True)
        for img in si: shutil.copy(img,od/img.name)
print('Partición lista (la partición NO cambia entre semillas; solo cambia la inicialización del modelo)')
for sn in ['train','val','test']:
    print(f'  {sn}:', {c: len(list((DEST/sn/c).glob("*"))) for c in ALL})

In [ ]:
import timm, numpy as np, time, json
import torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES=2; BATCH=32; EPOCHS=10; LR=1e-4; PATIENCE=4

# 6 originales + 3 modernas
ARCHS = {
    'EfficientNet-B4':   ('efficientnet_b4', 224),
    'ResNet50':          ('resnet50', 224),
    'DenseNet121':       ('densenet121', 224),
    'ViT-B/16':          ('vit_base_patch16_224', 224),
    'InceptionV3':       ('inception_v3', 299),
    'VGG16':             ('vgg16', 224),
    'ConvNeXt-Tiny':     ('convnext_tiny', 224),          # NUEVA
    'Swin-T':            ('swin_tiny_patch4_window7_224', 224),  # NUEVA
    'EfficientNetV2-S':  ('tf_efficientnetv2_s', 224),    # NUEVA
}

def get_loaders(img_size):
    train_tf=transforms.Compose([transforms.Resize((img_size,img_size)),transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(10),transforms.ColorJitter(brightness=0.15,contrast=0.15),
        transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    eval_tf=transforms.Compose([transforms.Resize((img_size,img_size)),transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    tr=datasets.ImageFolder('/content/dataset_bin/train',transform=train_tf)
    va=datasets.ImageFolder('/content/dataset_bin/val',transform=eval_tf)
    te=datasets.ImageFolder('/content/dataset_bin/test',transform=eval_tf)
    return (DataLoader(tr,batch_size=BATCH,shuffle=True,num_workers=2),
            DataLoader(va,batch_size=BATCH,shuffle=False,num_workers=2),
            DataLoader(te,batch_size=BATCH,shuffle=False,num_workers=2), tr.classes)

def build(tname):
    if tname=='inception_v3':
        return timm.create_model(tname,pretrained=True,num_classes=NUM_CLASSES,aux_logits=False)
    return timm.create_model(tname,pretrained=True,num_classes=NUM_CLASSES)

def train_eval(name, tname, img_size, seed, verbose=True):
    torch.manual_seed(seed); np.random.seed(seed)
    tr,va,te,classes=get_loaders(img_size)
    m=build(tname).to(device)
    crit=nn.CrossEntropyLoss(); opt=optim.AdamW(m.parameters(),lr=LR,weight_decay=0.01)
    sch=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
    best,bstate,noimp=0.0,None,0
    for ep in range(EPOCHS):
        t0=time.time(); m.train()
        for x,y in tr:
            x,y=x.to(device),y.to(device)
            opt.zero_grad(); crit(m(x),y).backward(); opt.step()
        sch.step(); m.eval(); vp,vt=[],[]
        with torch.no_grad():
            for x,y in va:
                vp.extend(m(x.to(device)).argmax(1).cpu().numpy()); vt.extend(y.numpy())
        vacc=accuracy_score(vt,vp)
        if verbose: print(f'  [{name} s{seed}] Ep {ep+1}/{EPOCHS} val={vacc:.4f} ({time.time()-t0:.0f}s)')
        if vacc>best: best=vacc; bstate={k:v.cpu().clone() for k,v in m.state_dict().items()}; noimp=0
        else:
            noimp+=1
            if noimp>=PATIENCE:
                if verbose: print(f'  [{name} s{seed}] Early stop ep {ep+1}')
                break
    if bstate: m.load_state_dict(bstate)
    # Evaluación en test
    m.eval(); preds,labels,probs=[],[],[]
    with torch.no_grad():
        for x,y in te:
            out=m(x.to(device)); probs.extend(torch.softmax(out,1).cpu().numpy())
            preds.extend(out.argmax(1).cpu().numpy()); labels.extend(y.numpy())
    preds,labels,probs=np.array(preds),np.array(labels),np.array(probs)
    cm=confusion_matrix(labels,preds); tn,fp=cm[0,0],cm[0,1]
    res={'accuracy':accuracy_score(labels,preds),
         'precision':precision_score(labels,preds,average='macro',zero_division=0),
         'recall_sensitivity':recall_score(labels,preds,average='macro',zero_division=0),
         'f1':f1_score(labels,preds,average='macro',zero_division=0),
         'specificity':float(tn/(tn+fp)) if (tn+fp)>0 else 0.0,
         'confusion_matrix':cm.tolist()}
    del m; torch.cuda.empty_cache()
    return res, probs, labels

## FASE A — Las 9 arquitecturas (semilla 42)

In [ ]:
fase_a, probs_a, test_labels = {}, {}, None
for name,(tname,isz) in ARCHS.items():
    print(f'\n{"="*55}\n{name}\n{"="*55}')
    r, pr, lb = train_eval(name, tname, isz, seed=42)
    fase_a[name]=r; probs_a[name]=pr
    if test_labels is None: test_labels=lb
    print(f'  >> TEST {name}: acc={r["accuracy"]*100:.2f}% f1={r["f1"]*100:.2f}%')

print('\n\nRANKING (semilla 42):')
for n,r in sorted(fase_a.items(), key=lambda x:-x[1]['accuracy']):
    print(f'  {n:<20} {r["accuracy"]*100:.2f}%')

## FASE B — Estabilidad: top-3 × 3 semillas

Mide si los resultados son estables o dependen del azar de la inicialización.

In [ ]:
SEEDS=[42,123,2024]
top3=[n for n,_ in sorted(fase_a.items(), key=lambda x:-x[1]['accuracy'])[:3]]
print(f'Top-3 a evaluar con {len(SEEDS)} semillas: {top3}\n')

estabilidad={n:{'accuracy':[], 'f1':[], 'specificity':[]} for n in top3}
# La semilla 42 ya la tenemos de la Fase A
for n in top3:
    estabilidad[n]['accuracy'].append(fase_a[n]['accuracy'])
    estabilidad[n]['f1'].append(fase_a[n]['f1'])
    estabilidad[n]['specificity'].append(fase_a[n]['specificity'])

for seed in SEEDS[1:]:  # 123 y 2024
    for n in top3:
        tname,isz=ARCHS[n]
        print(f'\n--- {n} | semilla {seed} ---')
        r,_,_ = train_eval(n, tname, isz, seed=seed, verbose=False)
        estabilidad[n]['accuracy'].append(r['accuracy'])
        estabilidad[n]['f1'].append(r['f1'])
        estabilidad[n]['specificity'].append(r['specificity'])
        print(f'  acc={r["accuracy"]*100:.2f}%')
print('\nFase B completa.')

In [ ]:
# Tabla de estabilidad: media ± desviación estándar
import pandas as pd
rows=[]
for n in top3:
    a=np.array(estabilidad[n]['accuracy'])*100
    f=np.array(estabilidad[n]['f1'])*100
    s=np.array(estabilidad[n]['specificity'])*100
    rows.append({'Arquitectura':n,
                 'Exactitud (media ± DE)':f'{a.mean():.2f} ± {a.std(ddof=1):.2f}',
                 'F1 (media ± DE)':f'{f.mean():.2f} ± {f.std(ddof=1):.2f}',
                 'Especificidad (media ± DE)':f'{s.mean():.2f} ± {s.std(ddof=1):.2f}',
                 'Rango exactitud':f'{a.min():.2f}–{a.max():.2f}'})
df_est=pd.DataFrame(rows)
print('ESTABILIDAD ENTRE SEMILLAS (n = 3: 42, 123, 2024)\n')
print(df_est.to_string(index=False))
df_est.to_csv('/content/estabilidad_semillas.csv',index=False)

print('\nValores individuales por semilla:')
for n in top3:
    vals=[f'{v*100:.2f}' for v in estabilidad[n]['accuracy']]
    print(f'  {n:<20} {" | ".join(vals)}')

## FASE C — Ensamblado y guardado

In [ ]:
from itertools import product
P=[probs_a[n] for n in top3]
best_acc,best_w=0,None
for w1,w2 in product(np.arange(0,1.05,0.1),repeat=2):
    w3=1-w1-w2
    if w3<0: continue
    ep=w1*P[0]+w2*P[1]+w3*P[2]
    a=accuracy_score(test_labels,ep.argmax(1))
    if a>best_acc: best_acc=a; best_w=(float(round(w1,2)),float(round(w2,2)),float(round(w3,2)))
w1,w2,w3=best_w; ep=w1*P[0]+w2*P[1]+w3*P[2]; epred=ep.argmax(1)
cm=confusion_matrix(test_labels,epred); tn,fp=cm[0,0],cm[0,1]
fase_a['Ensamblado (Top-3 ponderado)']={
    'accuracy':accuracy_score(test_labels,epred),
    'precision':precision_score(test_labels,epred,average='macro',zero_division=0),
    'recall_sensitivity':recall_score(test_labels,epred,average='macro',zero_division=0),
    'f1':f1_score(test_labels,epred,average='macro',zero_division=0),
    'specificity':float(tn/(tn+fp)),'confusion_matrix':cm.tolist()}
print(f'Ensamblado (Top-3: {top3}, pesos {best_w}): {best_acc*100:.2f}%')

In [ ]:
# Tabla final de las 9 arquitecturas + ensamblado
rows=[{'Modelo':n,'Exactitud (%)':round(r['accuracy']*100,2),'Precision (%)':round(r['precision']*100,2),
       'Sensibilidad (%)':round(r['recall_sensitivity']*100,2),'Especificidad (%)':round(r['specificity']*100,2),
       'F1 (%)':round(r['f1']*100,2)} for n,r in fase_a.items()]
df=pd.DataFrame(rows).sort_values('Exactitud (%)',ascending=False).reset_index(drop=True)
print(df.to_string(index=False))
df.to_csv('/content/resultados_9arqs.csv',index=False)

with open('/content/resumen_9arqs.json','w') as f: json.dump(fase_a,f,indent=2)
with open('/content/estabilidad_semillas.json','w') as f:
    json.dump({n:{k:[float(x) for x in v] for k,v in d.items()} for n,d in estabilidad.items()},f,indent=2)

files.download('/content/resultados_9arqs.csv')
files.download('/content/resumen_9arqs.json')
files.download('/content/estabilidad_semillas.csv')
files.download('/content/estabilidad_semillas.json')
print('\nListo. Pasame los 4 archivos.')